# Практическое занятие № 19. Нейрон, MLP, forward, BCE и backpropagation
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**  
Автор учебных материалов: В. А. Мазур, доцент ШКН ТюмГУ.

**Результат:** (1) реализовать прямой и обратный проход вручную на NumPy; (2) проверить градиенты численно; (3) обучить собственную MLP на учебном кейсе заказов; (4) сравнить с DummyClassifier. Студент обязан объяснить, откуда берутся градиенты и почему нельзя получать параметры нормализации из validation/test.

Файл `orders_dirty.csv` можно положить рядом с блокнотом. Для проверки запуска, если он недоступен, используйте приложенный `demo_orders_19_20.csv` — **отдельный синтетический** набор. В Google Colab загрузите нужные CSV на вкладке Files. Все результаты для демонабора являются только учебными.

## 1. Проверка окружения (0–7 мин)
Проверьте версии библиотек и установите случайное зерно. В графике нельзя выдавать демоданные за исходный набор студентов.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
print('NumPy:',np.__version__,'pandas:',pd.__version__)
SEED=42
rng=np.random.default_rng(SEED)

## 2. Одиночный искусственный нейрон (7–17 мин)
\(z=x^\top w+b\), \(a=\sigma(z)=1/(1+e^{-z})\).

**Задание.** Для \(x=(2,1)\), \(w=(0.4,-0.2)\), \(b=-0.1\) вычислите \(z\) вручную, затем сравните с кодом. Объясните, почему нелинейность нужна между слоями.

In [ ]:
def sigmoid(z):
    z=np.asarray(z,dtype=float)
    out=np.empty_like(z)
    positive=z>=0
    out[positive]=1/(1+np.exp(-z[positive]))
    ez=np.exp(z[~positive]); out[~positive]=ez/(1+ez)
    return out
x=np.array([2.,1.]); w=np.array([.4,-.2]); b=-.1
z=float(x@w+b);a=float(sigmoid(np.array([z]))[0])
print('z=',z,'activation=',a)
assert np.isclose(z,0.5)

## 3. Пишем MLP без фреймворка (17–37 мин)
Архитектура: 2 входа → скрытый слой 6 нейронов с tanh → 1 выходной нейрон с sigmoid. 

\(Z_1=XW_1+b_1\), \(A_1=\tanh(Z_1)\), \(Z_2=A_1W_2+b_2\), \(P=\sigma(Z_2)\).  
\(L=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]\).

При BCE+sigmoid \(\partial L/\partial Z_2=(P-Y)/n\). Далее применяем цепное правило. **Запрещено** заменять ручные производные на `torch.autograd` — это тема занятия 20.

In [ ]:
def init_mlp(n_inputs=2,n_hidden=6,seed=42):
    r=np.random.default_rng(seed)
    return {'W1':r.normal(0,0.5,(n_inputs,n_hidden)),
            'b1':np.zeros((1,n_hidden)),
            'W2':r.normal(0,0.5,(n_hidden,1)),
            'b2':np.zeros((1,1))}

def forward(X,p):
    z1=X@p['W1']+p['b1']
    a1=np.tanh(z1)
    z2=a1@p['W2']+p['b2']
    yhat=sigmoid(z2)
    return yhat,{'X':X,'A1':a1,'Z1':z1,'Z2':z2}

def binary_loss(y,p):
    q=np.clip(p,1e-9,1-1e-9)
    return float(-np.mean(y*np.log(q)+(1-y)*np.log(1-q)))

def backward(y,yhat,cache,p):
    n=len(y)
    dz2=(yhat-y)/n
    dw2=cache['A1'].T@dz2
    db2=dz2.sum(axis=0,keepdims=True)
    dz1=(dz2@p['W2'].T)*(1-cache['A1']**2)
    dw1=cache['X'].T@dz1
    db1=dz1.sum(axis=0,keepdims=True)
    return {'W1':dw1,'b1':db1,'W2':dw2,'b2':db2}

X_xor=np.array([[0,0],[0,1],[1,0],[1,1]],dtype=float)
y_xor=np.array([[0],[1],[1],[0]],dtype=float)
p=init_mlp()
yhat,cache=forward(X_xor,p)
grad=backward(y_xor,yhat,cache,p)
print('Слои:',{k:v.shape for k,v in p.items()})
print('loss before:',binary_loss(y_xor,yhat))

## 4. Проверка градиента конечными разностями (37–49 мин)
Для каждого выбранного веса: \(g_{num}=[L(\theta+\varepsilon)-L(\theta-\varepsilon)]/(2\varepsilon)\). Сравните с аналитическим \(g_{analytical}\). Малое расхождение подтверждает корректность реализации, но не гарантирует правильность постановки ML-задачи.

In [ ]:
def finite_diff(param,idx,X,y,p,eps=1e-5):
    old=p[param][idx]
    p[param][idx]=old+eps; lp=binary_loss(y,forward(X,p)[0])
    p[param][idx]=old-eps; lm=binary_loss(y,forward(X,p)[0])
    p[param][idx]=old
    return (lp-lm)/(2*eps)
checks=[('W1',(0,0)),('W1',(1,2)),('b1',(0,1)),('W2',(2,0)),('b2',(0,0))]
rows=[]
for name,idx in checks:
    gn=finite_diff(name,idx,X_xor,y_xor,p)
    ga=float(grad[name][idx]);rel=abs(gn-ga)/max(1e-8,abs(gn)+abs(ga))
    rows.append([name,str(idx),gn,ga,rel])
res=pd.DataFrame(rows,columns=['tensor','index','numeric','analytic','relative_diff'])
display(res)
assert (res.relative_diff < 1e-5).all(), 'Градиенты не совпали: проверьте backprop' 

## 5. Обучение на XOR и график потерь (49–59 мин)
Это демонстрация необходимости скрытого нелинейного слоя. **Задание:** попробуйте убрать `tanh` и прокомментируйте, что меняется.

In [ ]:
p=init_mlp(seed=7)
losses=[]
for epoch in range(4000):
    pred,c=forward(X_xor,p)
    g=backward(y_xor,pred,c,p)
    for k in p:
        p[k] -= 0.45*g[k]
    if epoch % 20 == 0:
        losses.append(binary_loss(y_xor,forward(X_xor,p)[0]))
print(pd.DataFrame({'x1':X_xor[:,0],'x2':X_xor[:,1],'target':y_xor[:,0],'prob':forward(X_xor,p)[0][:,0]}))
fig,ax=plt.subplots(figsize=(7,3));ax.plot(np.arange(len(losses))*20,losses)
ax.set(title='XOR: сходимость ручной MLP',xlabel='Итерация',ylabel='BCE');ax.grid(alpha=.3);plt.show()

## 6. Перенос ручной MLP на заказы (59–82 мин)
**Задание:** прогнозировать `late` по двум ранним числовым признакам `distance_km` и `queue_size`. Это упрощённая модель для проверки механики нейросети, не полноценное производственное решение. `actual_minutes`, `late`, `order_id` и сырой `created_at` в X не передавать.

Последовательность: аудит метки → исключение точного дубликата/противоречивой исторической метки → сортировка по времени → 60/20/20 → `SimpleImputer` и `StandardScaler`, **fit только train** → MLP на train → сравнение с DummyClassifier на validation. Test до выбора модели не открываем.

In [ ]:
dataset=Path('orders_dirty.csv') if Path('orders_dirty.csv').exists() else Path('demo_orders_19_20.csv')
if not dataset.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv или demo_orders_19_20.csv в рабочую папку Colab')
df=pd.read_csv(dataset,encoding='utf-8-sig');print('Файл:',dataset.name,'Размер:',df.shape)
needed={'order_id','created_at','distance_km','queue_size','promised_minutes','actual_minutes','late'}
assert needed<=set(df.columns),f'Отсутствуют поля: {needed-set(df.columns)}'
df=df.drop_duplicates().copy()
for c in ['distance_km','queue_size','promised_minutes','actual_minutes','late']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
df['created_at']=pd.to_datetime(df['created_at'],errors='coerce')
df=df.dropna(subset=['created_at','late']).copy()
df=df[df['late'].isin([0,1])].copy()
audit=(df['actual_minutes'].notna() & df['promised_minutes'].notna())
inconsistent=(df['late']!=((df['actual_minutes']-df['promised_minutes'])>15).astype(int)) & audit
print('Конфликтные метки (исключены):',int(inconsistent.sum()))
df=df.loc[~inconsistent].drop_duplicates(subset=['order_id'],keep='first').sort_values('created_at').reset_index(drop=True)
n=len(df);ntr=int(n*.6);nval=int(n*.2)
assert ntr>=12 and nval>=6 and n-ntr-nval>=6,'Для учебного разбиения требуется достаточное число заказов'
tr=df.iloc[:ntr];va=df.iloc[ntr:ntr+nval];te=df.iloc[ntr+nval:]
assert not (set(tr.order_id)&set(va.order_id)|set(tr.order_id)&set(te.order_id)|set(va.order_id)&set(te.order_id))
cols=['distance_km','queue_size']
imp=SimpleImputer(strategy='median');sc=StandardScaler()
Xtr=sc.fit_transform(imp.fit_transform(tr[cols]))
Xva=sc.transform(imp.transform(va[cols]))
ytr=tr['late'].to_numpy(dtype=float).reshape(-1,1)
yva=va['late'].to_numpy(dtype=float).reshape(-1,1)
assert set(np.unique(ytr.ravel()))=={0,1},'Нужны оба класса в train'
print('Train / Validation / Test:',len(tr),len(va),len(te))

In [ ]:
manual=init_mlp(seed=42)
htr,hva=[],[]
for step in range(900):
    pt,cache=forward(Xtr,manual)
    grads=backward(ytr,pt,cache,manual)
    for k in manual:
        manual[k] -= 0.08*grads[k]
    if step%15==0:
        htr.append(binary_loss(ytr,forward(Xtr,manual)[0]))
        hva.append(binary_loss(yva,forward(Xva,manual)[0]))
probs=forward(Xva,manual)[0].ravel()
pred=(probs>=.5).astype(int)
dummy=DummyClassifier(strategy='most_frequent').fit(Xtr,ytr.ravel())
base=dummy.predict(Xva)
print(pd.DataFrame([
    {'model':'Dummy (validation)','F1':f1_score(yva.ravel(),base,zero_division=0),'Accuracy':accuracy_score(yva.ravel(),base)},
    {'model':'NumPy MLP (validation)','F1':f1_score(yva.ravel(),pred,zero_division=0),'Accuracy':accuracy_score(yva.ravel(),pred)}
]))
fig,ax=plt.subplots(figsize=(7,3));steps=np.arange(len(htr))*15
ax.plot(steps,htr,label='train');ax.plot(steps,hva,label='validation')
ax.set(xlabel='Итерация',ylabel='BCE',title='Ручная MLP: потери на разных выборках');ax.grid(alpha=.3);ax.legend();plt.show()

## 7. Самостоятельная часть и сдача (82–90 мин)
1. Измените ширину скрытого слоя с 6 до 3 и 12, сравните графики на **validation** (не test).
2. Проверьте аналитические градиенты как минимум в пяти точках (укажите максимальную относительную ошибку).
3. Объясните в 5–7 предложениях: что означает BCE, где используется цепное правило и почему градиенты нельзя вычислять из округлённых классов.
4. Представьте `19_student.ipynb`, отчёт `Группа_Фамилия_З19_Отчет.docx` (2–4 страницы) с формулами, таблицами, двумя графиками и ограничениями.

**Оценивание (10 баллов):** нейрон/формулы — 2; правильный forward/backprop — 3; gradient check — 2; эксперимент на заказах и baseline — 2; отчёт и воспроизводимость — 1.

**Контрольные вопросы:** Зачем нужна нелинейность? Чем логиты отличаются от вероятностей? Почему среднее берётся по всем объектам? Что случится, если масштабировать train+validation вместе?